# Carbonara — What's New: a functionality guide

This notebook is a reference for everything added/changed in the fitting engine and tooling in this round of work, with brief explanations and runnable/illustrative usage for each. It's a companion to the existing `runCarbonaraMultimer.ipynb`-style notebooks and `carbonara_usage_guide.md` (which cover the *core* setup → launch → collect → analyze pipeline) — this one only covers what's **new**.

**Contents**
1. [Fit objective: proportional chi2/penalty weighting](#1)
2. [Distance constraints: hard / soft / capped / ensemble-OR](#2)
3. [Disulfide bond detection](#3)
4. [Ensemble writhe-difference restraint](#4)
5. [Console output & JSON log fields](#5)
6. [Internals: `FitMode` / `computeOverallFit` (for context, not day-to-day use)](#6)
7. [Regression test suite](#7)
8. [Quick reference: `RunMe_<name>.sh` argv map](#8)
9. [Bug fixes worth knowing about](#9)

Everything here is **opt-in and backward compatible**: any existing `RunMe_*.sh` / notebook / constraint file from before this work still runs with identical behaviour unless you deliberately use one of the new flags or file columns. The one exception is noted in §1 — it's a deliberate, permanent change with no revert switch.

<a id='1'></a>
## 1. Fit objective: proportional chi2/penalty weighting

**What changed:** how the search's single scalar objective ("Combined Fit" in the console, `ScatterFitFirst` in the JSON log) combines chi2 with the soft penalties (overlap, distance constraints, writhe, writhe-difference).

**Old behaviour:** penalties were added to chi2 more or less additively, so their influence on move-acceptance depended on the *absolute scale* of chi2 — negligible early in a hard fit or on a noisy/high-chi2 dataset (chi2 dominates), and only starting to matter once chi2 had already dropped near convergence.

**New behaviour:**

$$\text{currFit} = \chi^2 \times \big(1 + \text{penaltyWeight} \times \text{penalties}\big)$$

Proportional, not additive — penalties keep roughly the same *relative* influence on the objective throughout the whole search, regardless of how large or small chi2 currently is.

**How to use it:** `penaltyWeight` defaults to `5.0` and is tunable per system (there's no single correct value — depends on how many chains/constraints are active and how "wrong" a typical raw penalty sum is for your structure). Set it via `--penalty_weight` on `setup_carbonara_allAtom.py`:

```bash
python3 setup_carbonara_allAtom.py -p pdbFiles/structure.pdb -s saxsFiles/data.dat -n myRun \
    --rotation --penalty_weight 8.0
```

This is a **permanent change with no revert flag** — every run uses the proportional formula now, old and new alike. There's nothing to opt into here; only `penaltyWeight`'s *value* is tunable.

<a id='2'></a>
## 2. Distance constraints: hard / soft / capped / ensemble-OR

The `fixedDistanceConstraints<i>.dat` format grew from 5 columns to **9**. Every new column is optional and defaults to the old behaviour, so any constraint file written by the old code still works unchanged:

```
seg1  elem1  seg2  elem2  distance  tolerance  boundType  hard  ensembleOr
```

| Column | Default | Meaning |
|---|---|---|
| `seg1 elem1 seg2 elem2 distance` | *(required)* | Same as before: the two constrained points (Carbonara section index + element-within-section) and target distance. |
| `tolerance` | `0.5` | Smaller = stricter. Governs how quickly the penalty grows away from the target distance. |
| `boundType` | `0` | `0` = two-sided (penalise closer *or* farther than target). `1` = upper-bound-only (penalise only if farther — for restraints like crosslinks where being closer than the measured reach isn't a violation). |
| `hard` | `0` | `0` = **soft**: contributes a bounded penalty to the objective, can be traded off against chi2. `1` = **hard**: a true feasibility filter — a move that violates it beyond tolerance is rejected outright, no matter how much chi2 would improve. Excluded entirely from the soft-penalty sum. Use this for restraints that must never break (disulfides, a strict circular-permutation post) rather than relying on a very small tolerance, which only makes breaking it *expensive*, not impossible. |
| `ensembleOr` | `0` | Ensemble fitting only (`mixture_n > 1`). `0` = this pair's penalty is summed across every mixture state (every conformation pressured to satisfy it). `1` = **ensemble-OR**: only the single best/least-violating mixture state's penalty counts — "satisfied by any one conformation", e.g. a crosslink that might only form in one state of a solution ensemble. No effect on hard pairs, no effect at all when `mixture_n==1`. |

A single pair can't sensibly be both `hard=1` and `ensembleOr=1` — hard silently wins (enforced per-state, not via the ensemble-OR path), and the engine now prints a `WARNING` to stderr if it sees that combination, rather than letting it look like ensemble-OR is in effect when it isn't (see §9).

The soft-penalty cap (`distanceConstraintCap`, default `50.0`) applies per-pair to *soft* pairs only: identical to the old uncapped quartic for small violations, saturates instead of growing unboundedly for large ones — keeps a strict tolerance from swamping chi2 unpredictably. Set it via `--distance_constraint_cap` on the setup script. Hard pairs are unaffected (they're a feasibility filter, not a penalty).

### Generating a constraint file with the new columns

`CarbonaraDataTools.translate_distance_constraints` grew four new optional list parameters, one per new column, each defaulting to old behaviour if omitted:

```python
def translate_distance_constraints(
    contactPredsIn, coords, working_path,
    fixedDistList=[], toleranceList=[], boundTypeList=[], hardList=[], ensembleOrList=[]
)
```

In [ ]:
import CarbonaraDataTools as CDT
import numpy as np

# Example: three contact pairs (1-based residue indices), the first a strict
# disulfide-style constraint (hard), the other two ordinary soft contacts.
run_name = "myRun"
contactPreds = [[30, 126], [149, 205], [225, 591]]

coords = np.genfromtxt("carbonara_runs/" + run_name + "/coordinates1.dat")

CDT.translate_distance_constraints(
    contactPreds,
    coords,
    "carbonara_runs/" + run_name,
    fixedDistList=[],                 # [] -> measure distance from coords for every pair
    toleranceList=[0.2, 0.5, 0.5],    # tight tolerance on the hard pair, default on the rest
    boundTypeList=[0, 0, 0],
    hardList=[1, 0, 0],               # pair 0 is a hard feasibility constraint
    ensembleOrList=[0, 0, 0],
)

CDT.toggle_paired_predictions("RunMe_" + run_name + ".sh")

# Resulting file: carbonara_runs/myRun/fixedDistanceConstraints1.dat
# each row: seg1 elem1 seg2 elem2 distance tolerance boundType hard ensembleOr

Omitting a list entirely falls back to the old defaults for every pair — this is exactly what the *unmodified* Method-2 cell in `runCarbonaraMultimer.ipynb` still does:

```python
CDT.translate_distance_constraints(contactPreds, coords, working_path, fixedDistSet)
# -> every row gets tolerance=0.5, boundType=0, hard=0, ensembleOr=0
```

For an **ensemble** run (`mixture_n > 1`), remember `replicate_numbered_files` copies `fixedDistanceConstraints1.dat` to `fixedDistanceConstraints2.dat`, `...3.dat`, etc. — write the constraints once with the flags you want, then replicate; don't hand-edit the numbered files differently per state if you're relying on `ensembleOr`, since it depends on the same pair appearing in the same order in every state's file.

<a id='3'></a>
## 3. Disulfide bond detection

`CarbonaraDataTools.find_disulfide_bonds_from_pdb` scans the raw PDB for Cys SG–SG pairs at real disulfide-bond distance (default 1.8–2.5 Å, standard S–S length ≈2.05 Å), *before* any of Carbonara's own sanitizing/renumbering/chain-splitting.

In [ ]:
import CarbonaraDataTools as CDT

bonds = CDT.find_disulfide_bonds_from_pdb("pdbFiles/5_C239S.pdb")
bonds
# -> [((chain, resSeq), (chain, resSeq)), ...] using the PDB's own chain letters/residue numbers

This is a detection utility, not (yet) automatically wired into `setup_carbonara_allAtom.py` — you turn its output into hard distance constraints yourself, the same way as any other contact prediction (see §2), by mapping each `(chain, resSeq)` pair onto Carbonara's post-split Cα coordinates and residue indices, then calling `translate_distance_constraints(..., hardList=[1]*len(bonds))` with a tight `toleranceList`.

The companion function `disulfide_safe_linkers(varying_indices, pdb_path, coords_file, fingerprint_file, rand_dir='rand_structures', thr=1.5)` is used automatically inside `setup_carbonara_allAtom.py`'s linker-selection step: reshaping a loop rigidly repositions everything downstream of it (see `randomMol::reshapeMol`'s `for(int j=index+2; ...)` loop), so a disulfide bond can get stretched by reshaping a loop that sits between the two bonded Cys residues in sequence, even if the loop itself is nowhere near either one spatially. Rather than guess which loops those are from sequence position, it re-uses the same 25-sample-per-loop output already generated for the sheet-preservation check and empirically measures whether any sample stretched a known disulfide pair by more than `thr` Å — a loop is dropped from the varying set if it did. This runs automatically whenever `find_disulfide_bonds_from_pdb` finds any bonds in your input structure; nothing to configure unless you want a different `thr`.

For **backmapping** (all-atom reconstruction), pass a constraints file via `--disulfide_constraints_file` on `setup_carbonara_allAtom.py` — this gets threaded into the generated `RunMe_<name>.sh`'s background watcher (`watch_and_backmap.py --disulfide-file ...`), which uses it to write proper `SSBOND` records into the reconstructed all-atom PDB via `write_ssbond_records_to_pdb`.

<a id='4'></a>
## 4. Ensemble writhe-difference restraint

Ensemble fitting only (`mixture_n > 1`): softly penalises a candidate structure if the L1 distance between its full pairwise-writhe matrix and another ensemble member's exceeds a threshold — a way of discouraging two states in a mixture from collapsing onto near-identical folds when the data doesn't distinguish them.

**How to use it:** two flags on `setup_carbonara_allAtom.py`, both off by default (`<=0` disables the restraint entirely, so existing single-state and untouched ensemble runs are unaffected either way):

```bash
python3 setup_carbonara_allAtom.py -p pdbFiles/structure.pdb -s saxsFiles/data.dat -n myRun \
    --mixture_n 2 --max_writhe_diff 15.0 --writhe_diff_stride 2
```

| Flag | Meaning |
|---|---|
| `--max_writhe_diff` | Threshold above which the penalty kicks in. `<= 0` (default) disables it. |
| `--writhe_diff_stride` | Stride the Cα backbone by this many residues before computing writhe (e.g. `2` or `4`) — cheaper, and coarser: filters out local wiggle so the comparison reflects overall fold changes rather than backbone jitter. Only used when `--max_writhe_diff` is set. |

The resulting `WritheDiffPenalty` is folded into the same proportional objective as every other penalty (§1), and is also broken out separately in the JSON log (§5).

<a id='5'></a>
## 5. Console output & JSON log fields

**Console** (auto-colored when stdout is a real terminal, plain when redirected to a file — verified to emit zero ANSI escape bytes when piped):

```
----------------------------------- Initial Molecule -----------------------------------
Combined Fit   Chi2           Overlap Pen.   Writhe Pen.    Contact Pen.   Hard Constr.
5.6054         5.6054         0.0000         0.0000         0.0000         OK
------------------------------------------------------------------------------------------
```

and per fit-step during the search:

```
Improve Idx    Fit Step       Combined Fit   Chi2           Hard Constr.
5              46             0.6943         0.6259         OK
  Chi2 trend: ████████▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁  (0.6259 .. 0.6969)
```

- **Combined Fit** — the full proportional objective from §1 (`chi2 * (1 + penaltyWeight * penalties)`), what the search actually optimizes. **Chi2** below it is the unpenalized scattering agreement — watch *this* one to judge fit quality.
- **Hard Constr.** — `OK` (green) or `VIOLATED(<max violation>)` (red) for the current accepted state.
- **Chi2 trend** — a Unicode-block sparkline (▁▂▃▄▅▆▇█) over the last 30 *accepted* Chi2 values, min/max-scaled to that window, with the window's min/max printed alongside. Display-only — never written to the JSON log.

**JSON log** (`fitLog<i>.dat`, one object per line) — fields added this round:

```json
{
  "ImprovementIndex": 5, "FitStep": 46,
  "ScatterFitFirst": 0.6943,
  "Chi2": 0.6259,
  "HardConstraintsSatisfied": true, "HardConstraintsMaxViolation": 0.0,
  "WrithePenalty": 0.0, "OverlapPenalty": 0.0, "DistanceConstraints": 0.0,
  "WritheDiffPenalty": 0.0,
  "HydrationDensity": 1.2, "ElapsedTime(\u00b5s)": 812345,
  "KmaxCurr": 0.2,
  "ScatterPath": "...", "MoleculePath": "..."
}
```

`ScatterFitFirst` is (despite the name, kept for backward compatibility with existing log readers) the *combined* objective, not raw chi2 — use the separate `Chi2` field for the actual scattering fit. `HardConstraintsSatisfied`/`HardConstraintsMaxViolation` report only the *hard* pairs (which never contribute to `ScatterFitFirst`/`DistanceConstraints` at all). `WritheDiffPenalty` is the §4 restraint, `0.0` whenever it's disabled.

<a id='6'></a>
## 6. Internals: `FitMode` / `computeOverallFit`

*(For context if you're reading the C++ source or writing a new caller — nothing here changes how you invoke a run.)*

The 8 near-duplicated `getOverallFit*` functions in `moleculeFitAndState` (one per combination of "weighted chi2 or not" × "force inter-chain connection or not", each with a no-arg "baseline" and a `(molNew, i)` "trial move" version) were consolidated into two overloads parameterized by a small struct:

```cpp
struct FitMode { bool weightedChiSq = true; bool forceConnection = false; };

std::pair<double,double> computeOverallFit(
    experimentalData &ed, std::vector<std::vector<double>> &mixtureList,
    double &kmin, double &kmax, const FitMode &mode);              // baseline

std::pair<double,double> computeOverallFit(
    experimentalData &ed, std::vector<std::vector<double>> &mixtureList,
    ktlMolecule &molNew, double &kmin, double &kmax, int &i, const FitMode &mode); // trial move
```

This was a real correctness fix, not just cleanup: consolidating onto one formula fixed two live divergences that had crept in across the 8 copies —

1. One dead branch (`getOverallFitForceConnection`, non-ChiSq) was missing the `overlapPenalty` term that every other variant included (harmless today since every generated `RunMe_*.sh` hardcodes `useErrors="True"`, so that branch was never actually reached — but it would have silently activated if that default ever changed).
2. `increaseKmax` (called whenever the q-range widens mid-run) was hardcoded to always use the plain, non-weighted/non-connection-forcing formula, regardless of whether the run actually used `--rotation` or ChiSq-weighting — meaning every kmax-increase recomputation silently used a *different* formula than the rest of that run, for exactly one step. This was live in every real run and is now fixed by threading the caller's actual `FitMode` through instead.

Golden-value regression tests (§7) confirm the consolidated version matches the original 8-function behaviour exactly, aside from these two intentional fixes.

<a id='7'></a>
## 7. Regression test suite

New: `tests/regression_test.py`. Two layers:

- **Golden-value tests** (via `single_fit`, deterministic, step-0 only — no search randomness): baseline chi2, the soft-penalty cap, hard-constraint feasibility, ensemble-OR aggregation. Must match `tests/golden_values.json` exactly.
- **Invariant tests** (short live `predictStructureQvary` searches — stochastic, so exact trajectories aren't asserted, only properties that must always hold): a hard constraint is never violated in any *accepted* move; the soft-distance sum never exceeds its theoretical cap ceiling.

Fixtures live in `tests/fixtures/` (`MBP_apo_Bilbo_2.pdb`, `Saxs.dat`).

In [ ]:
# Run from the carbonara/ directory, after `cmake --build build`:
#   python3 tests/regression_test.py             # verify current build against tests/golden_values.json
#   python3 tests/regression_test.py --capture    # (re)write golden_values.json from the current build
#                                                  # -- use this to re-baseline right before a deliberate
#                                                  #    change to the fitting formula, not casually.
!python3 tests/regression_test.py

Expected output on a healthy build:

```
[ran] baseline_chi2: {'chi2': 203.68, 'scatter_first': 203.68}
[ran] soft_cap: {'distance_constraints_capped': 50}
[ran] hard_satisfied_initially: {'hard_satisfied': True, 'hard_max_violation': 0, 'distance_constraints': 0}
[ran] ensemble_or: {'or_on': 0, 'or_off': 50}
[OK] invariant: hard_never_violated_when_accepted
All regression checks passed.
```

If this fails with a Python import error (e.g. `scipy`/`liblapack` missing) rather than an actual value mismatch, that's an environment problem, not a regression — check your `CarbonaraDataTools.py` dependencies import cleanly first (`python3 -c "import CarbonaraDataTools"`).

<a id='8'></a>
## 8. Quick reference: `RunMe_<name>.sh` argv map

`setup_carbonara_allAtom.py` writes these as shell variables and passes them positionally to `predictStructureQvary`. argv[1]–[19] are unchanged from before; **argv[20]–[23] are new** (and, being appended at the end, don't disturb any older/dead caller like `main_multi.cpp` that only knows about the first 19):

| argv | Shell var | New flag | Default |
|---|---|---|---|
| 20 | `maxWritheDiff` | `--max_writhe_diff` | `-1.0` (disabled) |
| 21 | `writheDiffStride` | `--writhe_diff_stride` | `1` |
| 22 | `penaltyWeight` | `--penalty_weight` | `5.0` |
| 23 | `distanceConstraintCap` | `--distance_constraint_cap` | `50.0` |

All four are read defensively in `parameters.cpp::loadParameters` (`if (argc > N && strlen(argv[N]) > 0)`), so a script built before this round of work — which never passes argv[20]+ at all — gets the disabled/default values above and behaves exactly as it always did.

<a id='9'></a>
## 9. Bug fixes worth knowing about

A few things that were silently wrong before and are now fixed — worth knowing about if you ever compared numbers from before this round of work to after:

- **`increaseKmax` formula mismatch** (§6) — every kmax-increase step used to silently use a different fitting formula than the rest of the run. Now consistent.
- **`CARBONARA_DEBUG_FIT` diagnostic block ignored the hard-constraint gate** — its `willAccept=` line could say a move would be accepted when it was actually about to be rejected for violating a hard constraint. Debug-output-only; never affected actual search behaviour.
- **`disulfide_safe_linkers` could raise an unguarded `IndexError`** on malformed/edge-case coordinate input instead of falling back gracefully — now caught, falls back to returning `varying_indices` unfiltered (best-effort, as documented).
- **Hard + ensemble-OR conflict is now flagged**: setting both `hard=1` and `ensembleOr=1` on the same constraint pair used to silently do the "wrong" thing with no indication (see §2) — now prints a `WARNING` to stderr naming the pair.
- **A `std::length_error`/`EXC_BAD_ACCESS` crash in `randomMol::reshapeMol`** (`randomMolGen.cpp`) — pre-existing, not introduced by anything else here. Triggered when a freshly-regenerated *first* section of a chain came out length 1–2 and needed blending into the next section: the code read `molPos[index-1]` without checking `index > 0`, i.e. read one element before the start of the array. Fixed by adding the missing bounds guard (mirroring the identical guard already used everywhere else in that function) at all 6 vulnerable call sites; the search now just rejects that specific move (via the existing `suceeded=false` / `checkCalphas` rejection path) instead of crashing. Root-caused with a real backtrace from a from-scratch unoptimized (`-O0`) debug build, then verified fixed against the exact reproduction (IgG2 structure + `--rotation` + chain merge) across multiple runs with zero crashes.